In [ ]:

import re
from pathlib import Path
from typing import Dict, List, Tuple

import pandas as pd
import matplotlib.pyplot as plt
from matplotlib_venn import venn2

In [ ]:
GENE_LISTS_PATH = Path("<PATH_TO_GENE_LISTS_TSV>")

OUT_OVERLAPS_TSV = Path("<PATH_TO_GENE_PAIRWISE_OVERLAPS_TSV>")
OUT_PNG = Path("<PATH_TO_GENE_VENNS_PNG>")
OUT_SVG = Path("<PATH_TO_GENE_VENNS_SVG>")

AGE_LABELS = {
    "pre": "pre", 
    1: "< 50",
    2: "50 - 55",
    3: "56 - 60",
    4: "61 - 64",
    5: "65+",
}
SET_COLORS = ("#9DB4A5", "#5B7D8D")  
ALPHA = 0.7

def read_gene_lists(path):
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path.resolve()}")

    suf = path.suffix.lower()
    if suf in [".tsv", ".txt"]:
        df = pd.read_csv(path, sep="\t")
    elif suf == ".csv":
        df = pd.read_csv(path)
    elif suf in [".xlsx", ".xls"]:
        df = pd.read_excel(path)
    else:
        raise ValueError(f"Unsupported file extension: {suf} (use .tsv/.csv/.xlsx)")

    df = df.dropna(axis=1, how="all")

    for c in df.columns:
        df[c] = df[c].astype("string").str.strip()

    return df

gene_lists = read_gene_lists(GENE_LISTS_PATH)
print("Loaded gene-lists shape:", gene_lists.shape)
print("Columns:", list(gene_lists.columns))

def col_to_gene_set(df, col):
    if col not in df.columns:
        return set()
    s = (
        df[col]
        .dropna()
        .astype("string")
        .str.strip()
    )

    genes = {g for g in s.tolist() if g and g != "nan"}
    return genes

def build_column_sets(df):
    return {c: col_to_gene_set(df, c) for c in df.columns}

col_sets = build_column_sets(gene_lists)

def compute_pairwise_overlaps(col_sets, set[str]]):
    cols = list(col_sets.keys())
    rows = []
    for i in range(len(cols)):
        for j in range(i + 1, len(cols)):
            a, b = cols[i], cols[j]
            ov = sorted(col_sets[a].intersection(col_sets[b]))
            rows.append(
                {
                    "setA": a,
                    "setB": b,
                    "nA": len(col_sets[a]),
                    "nB": len(col_sets[b]),
                    "n_overlap": len(ov),
                    "overlap_genes": ",".join(ov),
                }
            )
    return pd.DataFrame(rows).sort_values(["setA", "setB"]).reset_index(drop=True)

overlaps_df = compute_pairwise_overlaps(col_sets)
print(overlaps_df[["setA", "setB", "nA", "nB", "n_overlap"]].head(20))

overlaps_df.to_csv(OUT_OVERLAPS_TSV, sep="\t", index=False)
print("Wrote:", OUT_OVERLAPS_TSV.resolve())

PAT = re.compile(r"^(SBP|DBP)_pre(?:_(\d+))?$", flags=re.IGNORECASE)

def find_trait_columns(df, trait):
    """
    Returns:
      pooled_col, [(bin_ix, colname), ...] sorted by bin_ix
    """
    pooled_col = None
    bins = []
    for c in df.columns:
        m = PAT.match(c)
        if not m:
            continue
        t = m.group(1).upper()
        if t != trait.upper():
            continue
        bin_ix = m.group(2)
        if bin_ix is None:
            pooled_col = c
        else:
            bins.append((int(bin_ix), c))

    bins = sorted(bins, key=lambda x: x[0])
    if pooled_col is None:
        raise ValueError(
            f"Could not find pooled column for {trait}. Expected something like '{trait}_pre'. "
            f"Found columns: {list(df.columns)}"
        )
    return pooled_col, bins

sbp_pooled, sbp_bins = find_trait_columns(gene_lists, "SBP")
dbp_pooled, dbp_bins = find_trait_columns(gene_lists, "DBP")

print("SBP pooled:", sbp_pooled, "SBP bins:", sbp_bins)
print("DBP pooled:", dbp_pooled, "DBP bins:", dbp_bins)

def plot_gene_venn_grid(
    df,
    output_png,
    output_svg = None,
    colors = SET_COLORS,
    alpha = ALPHA,
    font_size = 14,
):
    plt.rcParams.update({
        "font.size": font_size,
        "axes.titlesize": font_size + 2,
        "axes.labelsize": font_size,
        "xtick.labelsize": font_size - 1,
        "ytick.labelsize": font_size - 1,
    })

    sbp_pooled, sbp_bins = find_trait_columns(df, "SBP")
    dbp_pooled, dbp_bins = find_trait_columns(df, "DBP")

    # Use as many columns as needed for the max of SBP/DBP bins
    ncols = max(len(sbp_bins), len(dbp_bins), 1)
    nrows = 2
    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(6 * ncols, 10))
    if ncols == 1:
        axes = axes.reshape(2, 1)

    def _draw(ax, pooled_col, bin_col, title, bin_ix):
        A = col_to_gene_set(df, pooled_col)
        B = col_to_gene_set(df, bin_col)
        if len(A) == 0 and len(B) == 0:
            ax.axis("off")
            ax.text(0.5, 0.5, "no genes", ha="center", va="center")
            return

        venn2(
            [A, B],
            set_labels=(f"{pooled_col}", f"{bin_col}"),
            set_colors=colors,
            alpha=alpha,
            ax=ax,
        )
        age_label = AGE_LABELS.get(bin_ix, str(bin_ix))
        ax.set_title(f"{title}: pooled vs {age_label}")

    # Row 1: SBP
    for j in range(ncols):
        ax = axes[0, j]
        if j < len(sbp_bins):
            bin_ix, bin_col = sbp_bins[j]
            _draw(ax, sbp_pooled, bin_col, "SBP_pre genes", bin_ix)
        else:
            ax.set_visible(False)

    # Row 2: DBP
    for j in range(ncols):
        ax = axes[1, j]
        if j < len(dbp_bins):
            bin_ix, bin_col = dbp_bins[j]
            _draw(ax, dbp_pooled, bin_col, "DBP_pre genes", bin_ix)
        else:
            ax.set_visible(False)

    fig.tight_layout()
    fig.savefig(output_png, dpi=200, bbox_inches="tight")
    print("Saved:", output_png.resolve())

    if output_svg is not None:
        fig.savefig(output_svg, format="svg", bbox_inches="tight")
        print("Saved:", output_svg.resolve())

    plt.show()

plot_gene_venn_grid(
    gene_lists,
    output_png=OUT_PNG,
    output_svg=OUT_SVG,
    colors=SET_COLORS,
    alpha=ALPHA,
    font_size=14,
)

def pooled_vs_bins_overlap_table(df):
    rows = []
    for trait in ["SBP", "DBP"]:
        pooled_col, bins = find_trait_columns(df, trait)
        A = col_to_gene_set(df, pooled_col)
        for bin_ix, bin_col in bins:
            B = col_to_gene_set(df, bin_col)
            ov = sorted(A.intersection(B))
            rows.append({
                "trait": trait,
                "pooled_col": pooled_col,
                "bin_col": bin_col,
                "bin_ix": bin_ix,
                "bin_label": AGE_LABELS.get(bin_ix, str(bin_ix)),
                "n_pooled": len(A),
                "n_bin": len(B),
                "n_overlap": len(ov),
                "overlap_genes": ",".join(ov),
            })
    return pd.DataFrame(rows).sort_values(["trait", "bin_ix"]).reset_index(drop=True)

pooled_bin_df = pooled_vs_bins_overlap_table(gene_lists)
pooled_bin_df.to_csv("<PATH_TO_POOLED_VS_BINS_GENE_OVERLAPS_TSV>", sep="\t", index=False)
print("Wrote:", Path("<PATH_TO_POOLED_VS_BINS_GENE_OVERLAPS_TSV>").resolve())